# ДЗ №1. Задание 2
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

Рейтинги товаров Amazon.


In [6]:
import json
import os
import sys
from pyspark import SparkConf, SparkContext

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable

conf = SparkConf().setMaster("local[2]").setAppName("HW1-Amazon")
sc = SparkContext.getOrCreate(conf=conf)

## 1. Средний рейтинг товаров

Из каждого отзыва получаем `(asin, (оценка, 1))`. Через `reduceByKey` складываем оценки и количество отзывов отдельно для каждого товара. Затем делим сумму на количество.


In [17]:
def parse_review(line):
    review = json.loads(line)
    return review["asin"], (float(review["overall"]), 1)


def add_ratings(first, second):
    return first[0] + second[0], first[1] + second[1]


def mean_rating(totals):
    return totals[0] / totals[1]


ratings = sc.textFile("data/Electronics_5.json").map(parse_review)
average_ratings = ratings.reduceByKey(add_ratings).mapValues(mean_rating).cache()

print(f"Товаров с оценками: {average_ratings.count()}")
print(f"{'ASIN':<12} {'Рейтинг':>8}")
for asin, rating in average_ratings.sortByKey().take(10):
    print(f"{asin:<12} {rating:>8.2f}")


Товаров с оценками: 63001
ASIN          Рейтинг
0528881469       2.40
0594451647       4.20
0594481813       4.00
0972683275       4.46
1400501466       3.95
1400501520       4.40
1400501776       4.15
1400532620       3.61
1400532655       3.81
140053271X       3.83


## 2. Названия товаров

`leftOuterJoin` соединяет наборы по `asin` и сохраняет все товары с рассчитанным рейтингом.

In [18]:
def parse_product(line):
    product = eval(line, {"__builtins__": {}})
    return product["asin"], product.get("title")


def make_product(row):
    asin, (rating, title) = row
    return {"asin": asin, "title": title or "Без названия", "rating": rating}


def show_products(rows):
    print(f"{'ASIN':<12} {'Рейтинг':>8} Название")
    for product in rows:
        print(f"{product['asin']:<12} {product['rating']:>8.2f} {product['title']}")

def by_asin(product):
    return product["asin"]


titles = sc.textFile("data/meta_Electronics.json").map(parse_product)
products = average_ratings.leftOuterJoin(titles).map(make_product).cache()
show_products(products.takeOrdered(10, key=by_asin))


ASIN          Рейтинг Название
0528881469       2.40 Rand McNally 528881469 7-inch Intelliroute TND 700 Truck GPS
0594451647       4.20 Barnes &amp; Noble HDTV Adapter Kit for NOOK HD and NOOK HD+
0594481813       4.00 Barnes &amp; Noble OV/HB-ADP Universal Power Kit
0972683275       4.46 VideoSecu 24&quot; Long Arm TV Wall Mount Low Profile Full Motion Cantilever Swing &amp; Tilt wall bracket for most 22&quot; to 55&quot; LED LCD TV Monitor Flat Panel Screen VESA 200x200 400x400 up to 600x400mm - Articulating Arm Extend up to 24&quot; MAH
1400501466       3.95 Barnes &amp; Noble NOOK Tablet 16gb (Color, BNTV250)
1400501520       4.40 NOOK HD 7&quot; 16GB Tablet
1400501776       4.15 Barnes and Noble NOOK 7&quot; 8GB WiFi Tablet
1400532620       3.61 Barnes &amp; Noble Nook eReader - no 3G
1400532655       3.81 Barnes &amp; Noble Nook Color eReader
140053271X       3.83 Barnes &amp; Noble Nook Simple Touch eBook Reader (Wi-Fi Only)


## 3. Товары с рейтингом ниже 3

Фильтруем по среднему рейтингу без округления. Выводим 10 товаров с наименьшим рейтингом

In [20]:
def is_low_rated(product):
    return product["rating"] < 3


def by_rating(product):
    return product["rating"], product["asin"]


low_rated_products = products.filter(is_low_rated).cache()
print(f"Товаров с рейтингом ниже 3: {low_rated_products.count()}")
show_products(low_rated_products.takeOrdered(10, key=by_rating))


Товаров с рейтингом ниже 3: 2625
ASIN          Рейтинг Название
B00000JBIA       1.00 Agfa ePhoto SMILE 0.2MP Digital Camera
B000F1ORW6       1.00 GE 24746 Futura HDTV Ready Antenna
B000H13L4Y       1.00 ATI TV Wonder 200 PCI Video Card w/PVR Capabilities
B000NNFS4C       1.00 RCA DRC8335 DVD Recorder &amp; VCR Combo With Built-In Tuner
B00111JODG       1.00 StarTech HDMISPL1HH 1 feet Standard HDMI Cable - 1x HDMI (M) to 2x HDMI (F) (Discontinued by Manufacturer)
B0013WI5SS       1.00 Dynex-DX-AP100 Adapter Mini DVI to Mini-DIN
B001BLN91Y       1.00 Maxtor OneTouch 4 - 1 TB USB 2.0 Desktop External Hard Drive STM310005OTA3E1-RK
B001T9N0R6       1.00 Zeikos 57-in-1 USB 2.0 Flash Memory Card Reader ZE-CR201
B003KIQTXG       1.00 NEEWER&reg; Photographic Barn Door &amp; Honeycomb Grid &amp; Gel Set for Alienbees Alienbee Flash
B004GGRPGG       1.00 Rapid USB Charger Adapters include: Wall and Car + USB Cable for Samsung Galaxy TAB (P1000)


## 4. Сохранение результата

Сохраняем весь RDD товаров с рейтингом ниже 3.


In [ ]:
def product_to_json(product):
    return json.dumps(product, ensure_ascii=False)

low_rated_products.map(product_to_json).saveAsTextFile("output/low_rated_products")